## Learning notes — Py10. Transform Customers Data

### Learning objectives
Follow **Py10. Transform Customers Data** from its source inputs through its operations to the archived results.
- Explain join, schema, aggregate, json-array, profiling as used in this lesson.

### Source-specific learning focus
- Relate inputs by keys and explain the resulting cardinality.
- Compare raw fields with typed records and parsing failures.
- Identify grouping keys and the business grain of summary measures.
- Follow nested JSON and arrays as they become columns or rows.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `gizmobox.bronze.py_customers`, `gizmobox.silver.py_customers`.


# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Transform Customer Data
1. Remove records with NULL customer_id 
1. Remove exact duplicate records
1. Remove duplicate records based on created_timestamp
1. CAST the columns to the correct Data Type
1. Write transformed data to the Silver schema

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df = spark.sql('SELECT * FROM gizmobox.bronze.py_customers');
display(df)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df = spark.table('gizmobox.bronze.py_customers')
display(df)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df = spark.read.table('gizmobox.bronze.py_customers')
display(df)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

### 1. Remove records with NULL customer_id

## Learning notes — before source operation

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Exploding an array [a, b] yields two rows for that input row.

In [ ]:
df = spark.table('gizmobox.bronze.py_customers')
df_filtered = df.filter('customer_id is not null')
display (df_filtered)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

## Learning notes — before source operation

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Exploding an array [a, b] yields two rows for that input row.

In [ ]:
df_filtered = df.filter(df.customer_id.isNotNull())
display (df_filtered)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

In [ ]:
df_filtered = df.where(df.customer_id.isNotNull())
display (df_filtered)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-01 00:50:29	4858	Carla Morton	2004-06-21	joseph88@mail.com	2024-09-15	+1 8616454195
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-23 22:03:08	7207	Billy Scott	1997-03-17	christopher30@mail.com	2024-09-23	+1 5544387564
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-12 06:02:27	8539	Lori Mason	2002-11-01	stephanie7@mail.com	2024-09-12	+1 0498301620
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-24 13:03:13	9706	Jennifer Haas	2001-04-03	benjamin55@mail.com	2024-10-05	+1 4725460000
dbfs:/Volumes/giz

### 2. Remove exact duplicate records

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_distinct = df_filtered.distinct()
display(df_distinct)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-09 07:14:17	7123	Edgar Stevens	2003-02-15	kevin68@example.org	2024-10-27	+1 0793189847
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-21 13:20:26	8580	Shannon Austin	2002-03-22	john30@gmail.com	2024-10-07	+1 4594705629
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-19 14:03:09	5816	Amy Morales	1999-12-11	katelyn82@example.org	2024-11-04	+1 4140982719
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-07 22:20:00	2703	Anthony Miller	1997-11-02	None	2024-11-05	+1 1586310752
dbfs:/Volumes/gizmobox/landi

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_distinct = df_filtered.dropDuplicates()
display(df_distinct)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-09 07:14:17	7123	Edgar Stevens	2003-02-15	kevin68@example.org	2024-10-27	+1 0793189847
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-21 13:20:26	8580	Shannon Austin	2002-03-22	john30@gmail.com	2024-10-07	+1 4594705629
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-19 14:03:09	5816	Amy Morales	1999-12-11	katelyn82@example.org	2024-11-04	+1 4140982719
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-07 22:20:00	2703	Anthony Miller	1997-11-02	None	2024-11-05	+1 1586310752
dbfs:/Volumes/gizmobox/landi

### 3. Remove duplicate records based on created_timestamp

## Learning notes — before source operation

**aggregate**: Grouping changes the output grain to one row per group. Check null handling, counts versus distinct counts, and whether joins inflated measures before aggregation.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Grouping three orders for one customer by customer yields one customer-level group, not three order-level rows.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
from pyspark.sql import functions as F

df_max_ts = df_distinct.groupBy('customer_id') \
                       .agg(F.max('created_timestamp').alias('max_created_timestamp'))

display(df_max_ts)                       

Archived table:
customer_id	max_created_timestamp
1211	2024-12-16 16:54:10
1987	2024-11-16 01:47:25
2054	2024-11-03 17:22:56
2141	2024-12-17 15:05:15
2344	2024-10-26 15:14:33
2639	2024-12-21 07:08:28
2703	2024-11-07 22:20:00
3084	2024-12-20 12:32:29
3295	2025-01-17 07:50:48
3409	2024-10-02 14:53:40
3532	2024-12-26 09:29:16
3712	2024-11-24 11:14:03
3889	2025-01-25 13:28:31
3892	2024-11-14 20:41:18
4468	2024-11-23 18:16:18
4761	2024-11-14 16:41:14
4858	2024-10-01 00:50:29
4914	2024-11-12 12:57:38
4973	2025-01-26 09:46:03
4996	2025-01-12 20:13:31
5011	2024-10-15 02:48:29
5028	2024-10-06 19:55:52
5204	2024-11-09 14:52:31
5592	2024-12-04 13:16:03
5816	2024-11-19 14:03:09
5953	2025-01-17 15:59:39
6167	2025-01-14 18:31:28
6384	2024-12-24 14:46:04
6406	2024-12-13 14:23:27
6627	2024-10-22 05:52:57
6973	2024-12-25 11:06:35
7007	2025-01-05 17:29:34
7123	2024-11-09 07:14:17
7207	2024-10-23 22:03:08
7295	2024-11-08 02:26:42
7803	2024-11-09 20:24:30
7829	2024-12-07 18:09:58
7997	2024-12-22 23:00:48


## Learning notes — before source operation

**join**: Check join keys and cardinality on both inputs; unmatched keys and duplicate keys affect nulls and row counts in the joined output.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Two left rows and three right rows with the same matching key yield six joined rows.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_distinct_customer = (
                         df_distinct
                         .join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) & 
                                          (df_distinct.created_timestamp == df_max_ts.max_created_timestamp),
                                'inner')
                         .select(df_distinct['*'])                         
)
display(df_distinct_customer)

Archived table:
file_path	created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-09 07:14:17	7123	Edgar Stevens	2003-02-15	kevin68@example.org	2024-10-27	+1 0793189847
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-17 16:12:27	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json	2024-10-21 13:20:26	8580	Shannon Austin	2002-03-22	john30@gmail.com	2024-10-07	+1 4594705629
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-19 14:03:09	5816	Amy Morales	1999-12-11	katelyn82@example.org	2024-11-04	+1 4140982719
dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_11.json	2024-11-07 22:20:00	2703	Anthony Miller	1997-11-02	None	2024-11-05	+1 1586310752
dbfs:/Volumes/gizmobox/landi

### 4. CAST the column values to the correct data type

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
df_casted_customer = (
                        df_distinct_customer
                            .select(df_distinct_customer.created_timestamp.cast('timestamp'),
                                    df_distinct_customer.customer_id,
                                    df_distinct_customer.customer_name,
                                    df_distinct_customer.date_of_birth.cast('date'),
                                    df_distinct_customer.email,
                                    df_distinct_customer.member_since.cast('date'),
                                    df_distinct_customer.telephone)
)
display(df_casted_customer)

Archived table:
created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
2024-11-09T07:14:17Z	7123	Edgar Stevens	2003-02-15	kevin68@example.org	2024-10-27	+1 0793189847
2024-10-17T16:12:27Z	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
2024-10-21T13:20:26Z	8580	Shannon Austin	2002-03-22	john30@gmail.com	2024-10-07	+1 4594705629
2024-11-19T14:03:09Z	5816	Amy Morales	1999-12-11	katelyn82@example.org	2024-11-04	+1 4140982719
2024-11-07T22:20:00Z	2703	Anthony Miller	1997-11-02	None	2024-11-05	+1 1586310752
2025-01-03T03:28:34Z	9084	Olivia Lyons	1997-06-17	patty47@outlook.com	2024-12-16	+1 2909899135
2024-12-04T13:16:03Z	5592	Richard Neal	1995-11-05	eric52@gmail.com	2024-11-15	+1 4425187397
2024-11-09T14:52:31Z	5204	Jennifer Nichols DDS	1997-02-17	joseph8@example.org	2024-11-04	+1 0383356328
2024-11-16T01:47:25Z	1987	Amanda Alvarez	1995-10-11	eric40@outlook.com	2024-10-27	+1 9673066300
2024-11-03T17:22:56Z	2054	Alan Price	1998-08-25	conn

### 5. Write Data to a Delta Table

In [ ]:
df_casted_customer.writeTo('gizmobox.silver.py_customers').createOrReplace()

Archived result list: empty.

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.silver.py_customers;
```

In [ ]:
# Archived output only; no computation.

Archived table:
created_timestamp	customer_id	customer_name	date_of_birth	email	member_since	telephone
2024-11-09T07:14:17Z	7123	Edgar Stevens	2003-02-15	kevin68@example.org	2024-10-27	+1 0793189847
2024-10-17T16:12:27Z	9179	Richard Cox	1996-10-25	devon84@mail.com	2024-09-26	+1 6680703335
2024-10-21T13:20:26Z	8580	Shannon Austin	2002-03-22	john30@gmail.com	2024-10-07	+1 4594705629
2024-11-19T14:03:09Z	5816	Amy Morales	1999-12-11	katelyn82@example.org	2024-11-04	+1 4140982719
2024-11-07T22:20:00Z	2703	Anthony Miller	1997-11-02	None	2024-11-05	+1 1586310752
2025-01-03T03:28:34Z	9084	Olivia Lyons	1997-06-17	patty47@outlook.com	2024-12-16	+1 2909899135
2024-12-04T13:16:03Z	5592	Richard Neal	1995-11-05	eric52@gmail.com	2024-11-15	+1 4425187397
2024-11-09T14:52:31Z	5204	Jennifer Nichols DDS	1997-02-17	joseph8@example.org	2024-11-04	+1 0383356328
2024-11-16T01:47:25Z	1987	Amanda Alvarez	1995-10-11	eric40@outlook.com	2024-10-27	+1 9673066300
2024-11-03T17:22:56Z	2054	Alan Price	1998-08-25	conn